# 💥 Recursive Tree Growth

**One example — generating permutations — to see why brute force explodes and how pruning saves you.**

---

## 1. Concept Intuition

Imagine you're arranging 4 books on a shelf.

- For the 1st slot, you have **4 choices**
- For the 2nd slot, **3 remain**
- Then **2**, then **1**
- Total arrangements: $4 \times 3 \times 2 \times 1 = 24$

That's $4! = 24$. Not bad.

Now try 10 books: $10! = 3,628,800$. Twenty books: $20! \approx 2.4 \times 10^{18}$.

This is **combinatorial explosion**. The number of possibilities grows so fast that for even moderate $n$, exhaustive search is physically impossible.

### The recursion tree

Every recursive algorithm builds an implicit **tree**:
- **Root**: the initial problem
- **Branches**: the choices at each step
- **Leaves**: the base cases (complete solutions or dead ends)

The total work = total nodes visited. Understanding the tree's shape tells you the algorithm's cost.

### Backtracking = pruning

If you can detect *early* that a branch will fail (a partial solution can't lead to a valid answer), you **prune** it — don't explore its subtree. This is backtracking. The more you prune, the fewer nodes you visit.

## 2. Visual Representation

Let's visualize the full recursion tree for generating permutations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
%matplotlib inline

def build_perm_tree(elements, current=(), depth=0):
    """Build the recursion tree for permutations. Returns list of (node, parent, depth)."""
    node_label = str(current) if current else '()'
    nodes = [{'label': node_label, 'depth': depth, 'is_leaf': len(elements) == 0}]
    children = []
    
    for i, elem in enumerate(elements):
        remaining = elements[:i] + elements[i+1:]
        child_nodes = build_perm_tree(remaining, current + (elem,), depth + 1)
        children.append(child_nodes)
    
    return {'label': node_label, 'depth': depth, 'is_leaf': len(elements) == 0, 
            'children': children}

def draw_tree(tree, ax, x=0, y=0, span=1, parent_pos=None):
    """Recursively draw the tree."""
    color = '#10b981' if tree['is_leaf'] else '#6366f1'
    size = 8 if tree['is_leaf'] else 6
    
    ax.plot(x, y, 'o', color=color, markersize=size, zorder=3)
    
    if parent_pos is not None:
        ax.plot([parent_pos[0], x], [parent_pos[1], y], '-', color='#94a3b8', 
                linewidth=0.8, zorder=1)
    
    if tree['is_leaf']:
        ax.text(x, y - 0.3, tree['label'], ha='center', fontsize=6, color='#10b981')
    
    n_children = len(tree['children'])
    if n_children > 0:
        child_span = span / max(n_children, 1)
        start_x = x - span / 2 + child_span / 2
        for i, child_tree in enumerate(tree['children']):
            child_x = start_x + i * child_span
            draw_tree(child_tree, ax, child_x, y - 1, child_span * 0.9, (x, y))

# Visualize permutations of [1, 2, 3]
tree = build_perm_tree([1, 2, 3])

fig, ax = plt.subplots(figsize=(14, 6))
draw_tree(tree, ax, x=0, y=0, span=12)
ax.set_xlim(-7, 7)
ax.set_ylim(-4.5, 1)
ax.set_title('Recursion Tree: All Permutations of [1, 2, 3]', fontsize=14, fontweight='bold')
ax.text(-6.5, 0.5, f'3! = 6 leaves (permutations)\nTotal nodes: {1+3+6+6} = 16', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Mathematical Formulation

### Permutations

Number of ways to arrange $n$ objects:

$$P(n) = n! = n \times (n-1) \times (n-2) \times \cdots \times 1$$

### Combinations

Number of ways to choose $k$ objects from $n$ (order doesn't matter):

$$C(n, k) = \binom{n}{k} = \frac{n!}{k!(n-k)!}$$

### Recursion tree size

For a recursion with branching factor $b$ and depth $d$:

$$\text{Total nodes} = \frac{b^{d+1} - 1}{b - 1} \approx b^d \text{ (for large } b \text{)}$$

For permutations: branching factor decreases at each level ($n, n-1, \ldots, 1$), giving:

$$\text{Leaves} = n!, \quad \text{Total nodes} = \sum_{k=0}^{n} \frac{n!}{k!} \approx e \cdot n!$$

### Stirling's approximation

$$n! \approx \sqrt{2\pi n} \left(\frac{n}{e}\right)^n$$

This shows that $n!$ grows *much* faster than $2^n$ or even $n^n$.

## 4. Code Implementation

### Counting the explosion

In [ ]:
from math import factorial, comb

# Factorial growth vs exponential growth
n_vals = np.arange(1, 21)
factorials = np.array([float(factorial(n)) for n in n_vals])
exponentials = 2.0 ** n_vals
polynomials = n_vals ** 3.0

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogy(n_vals, polynomials, 'o-', label='n³', color='#10b981', linewidth=2)
ax.semilogy(n_vals, exponentials, 's-', label='2ⁿ', color='#f97316', linewidth=2)
ax.semilogy(n_vals, factorials, '^-', label='n!', color='#f43f5e', linewidth=2)

ax.set_xlabel('n', fontsize=12)
ax.set_ylabel('Operations (log scale)', fontsize=12)
ax.set_title('The Combinatorial Explosion: n! >> 2ⁿ >> n³', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Concrete numbers
print(f'{"n":>3} | {"n³":>12} | {"2^n":>15} | {"n!":>20}')
print('-' * 60)
for n in [5, 10, 15, 20]:
    print(f'{n:>3} | {n**3:>12,} | {2**n:>15,} | {factorial(n):>20,}')

In [ ]:
# Combinations: Pascal's triangle structure
max_n = 12
pascal = np.zeros((max_n, max_n))
for n in range(max_n):
    for k in range(n + 1):
        pascal[n][k] = comb(n, k)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap of C(n,k)
im = axes[0].imshow(pascal[:10, :10], cmap='YlOrRd', aspect='auto')
axes[0].set_xlabel('k'); axes[0].set_ylabel('n')
axes[0].set_title('C(n,k) — choosing k from n', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# C(n, k) for fixed n as k varies — the binomial bell
for n in [6, 10, 15, 20]:
    k_vals = np.arange(n + 1)
    c_vals = [comb(n, k) for k in k_vals]
    axes[1].plot(k_vals / n, c_vals, 'o-', label=f'n={n}', linewidth=1.5, markersize=4)

axes[1].set_xlabel('k/n (fraction selected)')
axes[1].set_ylabel('C(n,k)')
axes[1].set_title('Binomial coefficients peak at k=n/2', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Full expansion vs pruned expansion

In [ ]:
def count_nodes_full(n):
    """Count total nodes in the full permutation tree."""
    count = [0]
    def generate(remaining):
        count[0] += 1
        for i in range(len(remaining)):
            generate(remaining[:i] + remaining[i+1:])
    generate(list(range(n)))
    return count[0]

def count_nodes_pruned(n, constraint):
    """Count nodes with pruning: skip branches where constraint fails."""
    count = [0]
    def generate(current, remaining):
        count[0] += 1
        if not constraint(current):  # PRUNE!
            return
        for i in range(len(remaining)):
            generate(current + [remaining[i]], remaining[:i] + remaining[i+1:])
    generate([], list(range(n)))
    return count[0]

# Constraint: no two adjacent elements can be consecutive numbers
def no_adjacent_consecutive(current):
    """Returns False if the last two elements are consecutive."""
    if len(current) < 2:
        return True
    return abs(current[-1] - current[-2]) != 1

# Compare full vs pruned for different sizes
sizes = list(range(2, 9))
full_counts = [count_nodes_full(n) for n in sizes]
pruned_counts = [count_nodes_pruned(n, no_adjacent_consecutive) for n in sizes]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(np.array(sizes) - 0.15, full_counts, 0.3, label='Full tree', color='#f43f5e', alpha=0.8)
axes[0].bar(np.array(sizes) + 0.15, pruned_counts, 0.3, label='Pruned tree', color='#10b981', alpha=0.8)
axes[0].set_yscale('log')
axes[0].set_xlabel('n')
axes[0].set_ylabel('Nodes visited (log scale)')
axes[0].set_title('Full vs Pruned: Nodes Visited', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Savings percentage
savings = [(1 - p/f) * 100 for f, p in zip(full_counts, pruned_counts)]
axes[1].bar(sizes, savings, color='#6366f1', alpha=0.8)
axes[1].set_xlabel('n')
axes[1].set_ylabel('Nodes saved (%)')
axes[1].set_title('Pruning savings increase with n', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

for i, (s, v) in enumerate(zip(sizes, savings)):
    axes[1].text(s, v + 1, f'{v:.0f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f'{"n":>3} | {"Full":>10} | {"Pruned":>10} | {"Savings":>8}')
print('-' * 40)
for n, f, p in zip(sizes, full_counts, pruned_counts):
    print(f'{n:>3} | {f:>10,} | {p:>10,} | {(1-p/f)*100:>7.1f}%')

## 5. Interactive Experiment

### Recursion tree size explorer

In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider

@interact(
    branching=IntSlider(value=3, min=2, max=6, step=1, description='Branching'),
    depth=IntSlider(value=4, min=1, max=8, step=1, description='Depth'),
    prune_prob=FloatSlider(value=0.0, min=0.0, max=0.8, step=0.1, description='Prune %')
)
def tree_size_explorer(branching, depth, prune_prob):
    # Full tree node count
    full_nodes = sum(branching**d for d in range(depth + 1))
    
    # Simulate pruned tree
    np.random.seed(42)
    def count_pruned(d, max_d):
        if d > max_d:
            return 0
        count = 1  # this node
        for _ in range(branching):
            if np.random.random() > prune_prob:  # not pruned
                count += count_pruned(d + 1, max_d)
        return count
    
    pruned_nodes = count_pruned(0, depth)
    
    # Visualize tree levels
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    
    levels = list(range(depth + 1))
    full_per_level = [branching**d for d in levels]
    
    axes[0].barh(levels, full_per_level, color='#f43f5e', alpha=0.7, label='Full')
    axes[0].set_xlabel('Nodes at this level')
    axes[0].set_ylabel('Depth')
    axes[0].set_title(f'Full tree: {full_nodes:,} total nodes', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].grid(True, alpha=0.3)
    
    # Growth curve
    depths = np.arange(1, 12)
    totals = np.array([(branching**(d+1) - 1) / (branching - 1) for d in depths])
    axes[1].semilogy(depths, totals, 'o-', color='#6366f1', linewidth=2)
    axes[1].axvline(depth, color='#f97316', linestyle='--', alpha=0.5, label=f'Current depth={depth}')
    axes[1].set_xlabel('Depth')
    axes[1].set_ylabel('Total nodes (log scale)')
    axes[1].set_title(f'Tree growth (branching={branching})', fontsize=12, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f'Full tree:   {full_nodes:>10,} nodes')
    print(f'Pruned tree: {pruned_nodes:>10,} nodes (~{(1-pruned_nodes/full_nodes)*100:.1f}% saved)')
    print(f'If each node takes 1μs: full = {full_nodes/1e6:.2f}s, pruned = {pruned_nodes/1e6:.2f}s')

### Permutation vs combination counter

In [ ]:
from math import factorial, comb, perm

@interact(
    n=IntSlider(value=10, min=1, max=20, step=1, description='n (total)'),
    k=IntSlider(value=3, min=1, max=10, step=1, description='k (choose)')
)
def counting_explorer(n, k):
    k = min(k, n)
    
    p = perm(n, k)
    c = comb(n, k)
    total_perms = factorial(n)
    subsets = 2**n
    
    fig, ax = plt.subplots(figsize=(9, 5))
    
    values = [c, p, total_perms, subsets]
    labels = [f'C(n,k)\n{c:,}', f'P(n,k)\n{p:,}', f'n!\n{total_perms:,}', f'2ⁿ\n{subsets:,}']
    colors = ['#10b981', '#6366f1', '#f43f5e', '#f97316']
    
    bars = ax.bar(range(4), values, color=colors, alpha=0.8)
    ax.set_xticks(range(4))
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_yscale('log')
    ax.set_title(f'Counting: n={n}, k={k}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count (log scale)')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print(f'C({n},{k}) = {c:,} — ways to choose {k} from {n} (order irrelevant)')
    print(f'P({n},{k}) = {p:,} — ways to arrange {k} from {n} (order matters)')
    print(f'  Ratio: P/C = {p/c:.0f} = {k}! (the {k}! orderings within each combination)')

## 6. Tool Exploration

Understanding recursion and tree structures programmatically.

In [ ]:
from math import factorial, comb
from itertools import permutations, combinations

# Python's built-in combinatorics
print('=== itertools: lazy generators ===')
print(f'type(permutations([1,2,3])): {type(permutations([1,2,3]))}')
print(f'Permutations of [1,2,3]:')
for p in permutations([1, 2, 3]):
    print(f'  {p}')
print()

print(f'Combinations C(5,2):')
for c in combinations(range(5), 2):
    print(f'  {c}')
print()

# Verify counts
n, k = 10, 3
print(f'=== Verification ===')
print(f'comb({n},{k}) = {comb(n,k)}')
print(f'len(list(combinations(range({n}), {k}))) = {len(list(combinations(range(n), k)))}')
print(f'Match: {comb(n,k) == len(list(combinations(range(n), k)))}')
print()

# Key insight: generators are lazy — they don't compute all at once
print(f'=== Lazy evaluation ===')
gen = permutations(range(10))  # 10! = 3,628,800 permutations
print(f'Generator created (no computation yet): {type(gen)}')
print(f'First 3 permutations:')
for i, p in enumerate(gen):
    if i >= 3: break
    print(f'  {p}')

## 7. Reflection Questions

1. In the tree size explorer, set branching=2 and increase depth from 1 to 10. By what factor does the total node count grow each time you increase depth by 1? How does this relate to $2^d$?

2. Why does n! grow faster than 2ⁿ? At what value of n does n! first exceed 2ⁿ? Can you prove this mathematically?

3. The N-Queens problem places n queens on an n×n board so no two attack each other. For the brute force approach, how many total positions would you check? How does backtracking reduce this?

4. In the pruned permutation example, the constraint "no adjacent consecutive numbers" prunes increasing percentages as n grows. Why? What property of the constraint causes this?

5. Think about the Traveling Salesman Problem: visit all n cities with minimum total distance. The brute force tries all permutations of cities. For n=20, how many routes is that? Dynamic programming solves it in $O(n^2 \cdot 2^n)$ — is that better, and why is it still exponential?

---

*You've now built intuition for the four mathematical pillars of algorithm design: growth, structure, randomness, and combinatorics. Go build something.*

## 🔬 Break-It Lab
(Exercises merged from earlier structure)

In [ ]:
import numpy as np
from math import factorial, comb
from math101.testing import check_answer, check_complexity, difficulty_banner, score_report
from math101.tracker import record_score

results = []

## 🟢 Warm-Up: Recall & Predict

## 🟡 Challenge: Apply & Analyze

## 🔴 Boss Level: Implement From Scratch

## 📊 Results

In [ ]:
score_report(results)
record_score('Combinatorics_and_Explosions', 'exercises', sum(results), len(results))

## The Feynman Technique
Explain [this concept] in plain English to someone who has never seen it. Write it in the cell below. Then check: did you use any jargon you can't define from scratch? If yes, go back.

*(Write your explanation here...)*

## Review
- **Q:** 
- **A:** 

- **Q:** 
- **A:** 

## The Takeaway
> **The one thing to carry forward is:** *(Write the insight, not the formula)*